# 04 Compute UQ and Metrics

Objective: convert cached model outputs into UQ scores and paper-facing metrics, including calibration and modality-specific diagnostics.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, BENCHMARK_VARIANT


## Load Raw Outputs and Benchmark


In [ ]:
benchmark_path = eu.variant_path(PROJECT_ROOT / "data/processed/benchmark_items.csv", BENCHMARK_VARIANT)
raw_path = eu.variant_path(PROJECT_ROOT / "data/processed/model_outputs_raw.jsonl", BENCHMARK_VARIANT)

benchmark = eu.read_csv_rows(benchmark_path)
all_raw_rows = eu.read_jsonl(raw_path)
requested_run_id = os.getenv("RUN_ID") or os.getenv("ANALYSIS_RUN_ID")
run_prefix = "full" if BENCHMARK_VARIANT == "must" else f"full-{BENCHMARK_VARIANT}"
selected_run_id, raw_rows = eu.select_run_rows(all_raw_rows, run_id=requested_run_id, prefix=run_prefix)
print(f"Benchmark variant: {BENCHMARK_VARIANT}")
print(f"Benchmark path: {benchmark_path}")
print(f"Raw output path: {raw_path}")
print(f"Benchmark items: {len(benchmark)}")
print(f"Raw output rows: {len(all_raw_rows)}")
print(f"Selected run_id: {selected_run_id}")
print(f"Selected raw rows: {len(raw_rows)}")
if all_raw_rows and not raw_rows:
    available = sorted({row.get("run_id", "") for row in all_raw_rows if row.get("run_id")})
    raise ValueError(f"No rows found for selected run_id. Available run_ids: {available[-10:]}")


## Build UQ Scores


In [ ]:
scores = eu.build_uq_scores(benchmark, raw_rows)
baseline_scores = eu.build_rule_baseline_scores(benchmark)
scores.extend(baseline_scores)
scores_path = eu.variant_path(PROJECT_ROOT / "data/processed/uq_scores.csv", BENCHMARK_VARIANT)
eu.write_csv_rows(scores_path, scores)
print(f"Wrote UQ scores: {scores_path}")
print(f"Score rows: {len(scores)} including {len(baseline_scores)} rule-baseline rows")
scores[:3]


## Metric Summary


In [ ]:
summary = eu.metric_summary_by_model_task_method(scores)
summary_path = eu.variant_path(PROJECT_ROOT / "data/processed/metrics_summary.csv", BENCHMARK_VARIANT)
eu.write_csv_rows(summary_path, summary)
print(f"Wrote summary: {summary_path}")
fields = [
    "model",
    "task",
    "uq_method",
    "n",
    "accuracy",
    "f1_or_macro_f1",
    "over_commitment",
    "brier",
    "ece",
    "auroc",
    "monotonicity_violations",
    "high_conf_overcommit_80",
    "high_conf_overcommit_90",
    "text_modality_accuracy",
    "label_text_consistency",
    "text_over_commitment",
    "text_high_conf_overcommit_80",
    "text_high_conf_overcommit_90",
    "error_detection_auroc",
    "parse_failure_rate",
]
print(eu.markdown_table(summary, fields))


## Bootstrap Confidence Intervals Over Seeds


In [ ]:
ci_rows = []
for key, rows in eu.grouped(scores, ["model", "task", "uq_method"]).items():
    model, task, uq_method = key

    def acc_metric(sample_rows):
        return eu.accuracy_score(
            [int(row["y_true"]) for row in sample_rows],
            [int(row["y_pred"]) for row in sample_rows],
        )

    def brier_metric(sample_rows):
        return eu.brier_score(
            [int(row["y_true"]) for row in sample_rows],
            eu.calibration_probabilities(sample_rows, task),
        )

    acc_point, acc_low, acc_high = eu.bootstrap_seed_metric(rows, acc_metric, iterations=1000)
    brier_point, brier_low, brier_high = eu.bootstrap_seed_metric(rows, brier_metric, iterations=1000)
    ci_rows.append({
        "model": model,
        "task": task,
        "uq_method": uq_method,
        "accuracy": acc_point,
        "accuracy_ci_low": acc_low,
        "accuracy_ci_high": acc_high,
        "brier": brier_point,
        "brier_ci_low": brier_low,
        "brier_ci_high": brier_high,
    })

ci_path = eu.variant_path(PROJECT_ROOT / "data/processed/bootstrap_seed_ci.csv", BENCHMARK_VARIANT)
eu.write_csv_rows(ci_path, ci_rows)
print(f"Wrote bootstrap CIs: {ci_path}")
print(eu.markdown_table(ci_rows, ["model", "task", "uq_method", "accuracy", "accuracy_ci_low", "accuracy_ci_high", "brier", "brier_ci_low", "brier_ci_high"]))


## Sensitivity Check: Recommended Strength = 0.75


In [ ]:
benchmark_075 = []
for row in benchmark:
    row = dict(row)
    row["numeric_strength"] = eu.NUMERIC_STRENGTH_RECOMMENDED_075[row["source_modality"]]
    benchmark_075.append(row)

scores_075 = eu.build_uq_scores(benchmark_075, raw_rows)
scores_075.extend(eu.build_rule_baseline_scores(benchmark_075))
summary_075 = eu.metric_summary_by_model_task_method(scores_075)
sensitivity_path = eu.variant_path(PROJECT_ROOT / "data/processed/metrics_summary_recommended075.csv", BENCHMARK_VARIANT)
eu.write_csv_rows(sensitivity_path, summary_075)
print(f"Wrote sensitivity summary: {sensitivity_path}")
print(eu.markdown_table(summary_075, ["model", "task", "uq_method", "spearman_modality_p_yes"]))
